In [1]:
#load 
from pathlib import Path
import subprocess
import pandas as pd
import tempfile
import os

OpenSMILE, intensity 3 dB

In [ ]:
# paths
BASE_DIR = Path(r"C:\Users\marti\Documents\Technical Medicine Master\Stages TM2\TM2-3\Technische opdracht")

OPENSMILE_EXE = BASE_DIR / "tools" / "openSMILE" / "bin" / "SMILExtract.exe"
CONFIG_FILE = BASE_DIR / "tools" / "openSMILE" / "config" / "egemaps" / "v02" / "eGeMAPSv02.conf"

AUDIO_ROOT = BASE_DIR / "TAME" / "wiener_intensity_perturbations" / "+3dB"

OUTPUT_FEATURES_CSV = BASE_DIR / "opensmile_intensity_3dB_features.csv"
OUTPUT_FAILED_CSV = BASE_DIR / "opensmile_intensity_3dB_failed_files.csv"

# check
print("SMILExtract exists:", OPENSMILE_EXE.exists())
print("Config exists:", CONFIG_FILE.exists())
print("Audio root exists:", AUDIO_ROOT.exists())

if not OPENSMILE_EXE.exists():
    raise FileNotFoundError(f"SMILExtract.exe not found:\n{OPENSMILE_EXE}")

if not CONFIG_FILE.exists():
    raise FileNotFoundError(f"Config file not found:\n{CONFIG_FILE}")

if not AUDIO_ROOT.exists():
    raise FileNotFoundError(f"Audio root not found:\n{AUDIO_ROOT}")

#WAV files
wav_files = sorted(AUDIO_ROOT.rglob("*.wav"))
print(f"Number of wav. files found: {len(wav_files)}")

if len(wav_files) == 0:
    raise ValueError("no .wav files found.")

# Run OpenSMILE
all_rows = []
failed_files = []

for i, wav_file in enumerate(wav_files, start=1):
    print(f"[{i}/{len(wav_files)}] Processing: {wav_file.name}")

    participant_id = wav_file.parent.name
    filename = wav_file.name

    with tempfile.NamedTemporaryFile(suffix=".csv", delete=False) as tmp:
        temp_output_csv = Path(tmp.name)

    try:
        if temp_output_csv.exists():
            temp_output_csv.unlink(missing_ok=True)

        cmd = [
            str(OPENSMILE_EXE),
            "-C", str(CONFIG_FILE),
            "-I", str(wav_file),
            "-csvoutput", str(temp_output_csv),
        ]

        result = subprocess.run(cmd, capture_output=True, text=True)

        if result.returncode != 0:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": "openSMILE return code != 0",
                "stderr": result.stderr
            })
            continue

        if not temp_output_csv.exists():
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": "no output made",
                "stderr": result.stderr
            })
            continue

        try:
            feature_df = pd.read_csv(temp_output_csv, sep=";", engine="python")
        except Exception as e:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": f"error in csv: {e}",
                "stderr": result.stderr
            })
            continue

        if feature_df.empty:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": "Output csv is leeg",
                "stderr": result.stderr
            })
            continue

        #adding information 
        row = feature_df.iloc[0].to_dict()
        row["name"] = filename
        row["participant_id"] = participant_id
        row["filename"] = filename
        row["file_path"] = str(wav_file)

        all_rows.append(row)

    except Exception as e:
        failed_files.append({
            "participant_id": participant_id,
            "filename": filename,
            "file_path": str(wav_file),
            "reason": f"error {e}",
            "stderr": ""
        })

    finally:
        if temp_output_csv.exists():
            temp_output_csv.unlink(missing_ok=True)


#Save
features_df = pd.DataFrame(all_rows)

# Kolommen netjes ordenen
preferred_front_cols = ["participant_id", "filename", "file_path", "name", "frameTime"]
remaining_cols = [col for col in features_df.columns if col not in preferred_front_cols]
features_df = features_df[[col for col in preferred_front_cols if col in features_df.columns] + remaining_cols]

features_df.to_csv(OUTPUT_FEATURES_CSV, index=False)

print("\nKlaar.")
print("number of errors in features:", len(features_df))
print("numer of errors in the files:", len(failed_files))
print("Features saves in :")
print(OUTPUT_FEATURES_CSV)

if failed_files:
    failed_df = pd.DataFrame(failed_files)
    failed_df.to_csv(OUTPUT_FAILED_CSV, index=False)
    print("\n errors saved:")
    print(OUTPUT_FAILED_CSV)


# Results
print("\nShape features dataframe:", features_df.shape)
display(features_df.head())

OpenSMILE, intensity 6dB

In [ ]:
# paths
BASE_DIR = Path(r"C:\Users\marti\Documents\Technical Medicine Master\Stages TM2\TM2-3\Technische opdracht")

OPENSMILE_EXE = BASE_DIR / "tools" / "openSMILE" / "bin" / "SMILExtract.exe"
CONFIG_FILE = BASE_DIR / "tools" / "openSMILE" / "config" / "egemaps" / "v02" / "eGeMAPSv02.conf"

AUDIO_ROOT = BASE_DIR / "TAME" / "wiener_intensity_perturbations" / "+6dB"

OUTPUT_FEATURES_CSV = BASE_DIR / "opensmile_intensity_6dB_features.csv"
OUTPUT_FAILED_CSV = BASE_DIR / "opensmile_intensity_6dB_failed_files.csv"

# check
print("SMILExtract exists:", OPENSMILE_EXE.exists())
print("Config exists:", CONFIG_FILE.exists())
print("Audio root exists:", AUDIO_ROOT.exists())

if not OPENSMILE_EXE.exists():
    raise FileNotFoundError(f"SMILExtract.exe not found:\n{OPENSMILE_EXE}")

if not CONFIG_FILE.exists():
    raise FileNotFoundError(f"Config file not found:\n{CONFIG_FILE}")

if not AUDIO_ROOT.exists():
    raise FileNotFoundError(f"Audio root not found:\n{AUDIO_ROOT}")

#WAV files
wav_files = sorted(AUDIO_ROOT.rglob("*.wav"))
print(f"Number of wav. files found: {len(wav_files)}")

if len(wav_files) == 0:
    raise ValueError("no .wav files found.")

# Run OpenSMILE
all_rows = []
failed_files = []

for i, wav_file in enumerate(wav_files, start=1):
    print(f"[{i}/{len(wav_files)}] Processing: {wav_file.name}")

    participant_id = wav_file.parent.name
    filename = wav_file.name

    with tempfile.NamedTemporaryFile(suffix=".csv", delete=False) as tmp:
        temp_output_csv = Path(tmp.name)

    try:
        if temp_output_csv.exists():
            temp_output_csv.unlink(missing_ok=True)

        cmd = [
            str(OPENSMILE_EXE),
            "-C", str(CONFIG_FILE),
            "-I", str(wav_file),
            "-csvoutput", str(temp_output_csv),
        ]

        result = subprocess.run(cmd, capture_output=True, text=True)

        if result.returncode != 0:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": "openSMILE return code != 0",
                "stderr": result.stderr
            })
            continue

        if not temp_output_csv.exists():
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": "no output made",
                "stderr": result.stderr
            })
            continue

        try:
            feature_df = pd.read_csv(temp_output_csv, sep=";", engine="python")
        except Exception as e:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": f"error in csv: {e}",
                "stderr": result.stderr
            })
            continue

        if feature_df.empty:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": "Output csv is leeg",
                "stderr": result.stderr
            })
            continue

        #adding information 
        row = feature_df.iloc[0].to_dict()
        row["name"] = filename
        row["participant_id"] = participant_id
        row["filename"] = filename
        row["file_path"] = str(wav_file)

        all_rows.append(row)

    except Exception as e:
        failed_files.append({
            "participant_id": participant_id,
            "filename": filename,
            "file_path": str(wav_file),
            "reason": f"error {e}",
            "stderr": ""
        })

    finally:
        if temp_output_csv.exists():
            temp_output_csv.unlink(missing_ok=True)


#Save
features_df = pd.DataFrame(all_rows)

# Kolommen netjes ordenen
preferred_front_cols = ["participant_id", "filename", "file_path", "name", "frameTime"]
remaining_cols = [col for col in features_df.columns if col not in preferred_front_cols]
features_df = features_df[[col for col in preferred_front_cols if col in features_df.columns] + remaining_cols]

features_df.to_csv(OUTPUT_FEATURES_CSV, index=False)

print("\nKlaar.")
print("number of errors in features:", len(features_df))
print("numer of errors in the files:", len(failed_files))
print("Features saves in :")
print(OUTPUT_FEATURES_CSV)

if failed_files:
    failed_df = pd.DataFrame(failed_files)
    failed_df.to_csv(OUTPUT_FAILED_CSV, index=False)
    print("\n errors saved:")
    print(OUTPUT_FAILED_CSV)


# Results
print("\nShape features dataframe:", features_df.shape)
display(features_df.head())

OpenSMILE, intensity -3dB

In [ ]:
# paths
BASE_DIR = Path(r"C:\Users\marti\Documents\Technical Medicine Master\Stages TM2\TM2-3\Technische opdracht")

OPENSMILE_EXE = BASE_DIR / "tools" / "openSMILE" / "bin" / "SMILExtract.exe"
CONFIG_FILE = BASE_DIR / "tools" / "openSMILE" / "config" / "egemaps" / "v02" / "eGeMAPSv02.conf"

AUDIO_ROOT = BASE_DIR / "TAME" / "wiener_intensity_perturbations" / "-3dB"

OUTPUT_FEATURES_CSV = BASE_DIR / "opensmile_intensity_-3dB_features.csv"
OUTPUT_FAILED_CSV = BASE_DIR / "opensmile_intensity_-3dB_failed_files.csv"

# check
print("SMILExtract exists:", OPENSMILE_EXE.exists())
print("Config exists:", CONFIG_FILE.exists())
print("Audio root exists:", AUDIO_ROOT.exists())

if not OPENSMILE_EXE.exists():
    raise FileNotFoundError(f"SMILExtract.exe not found:\n{OPENSMILE_EXE}")

if not CONFIG_FILE.exists():
    raise FileNotFoundError(f"Config file not found:\n{CONFIG_FILE}")

if not AUDIO_ROOT.exists():
    raise FileNotFoundError(f"Audio root not found:\n{AUDIO_ROOT}")

#WAV files
wav_files = sorted(AUDIO_ROOT.rglob("*.wav"))
print(f"Number of wav. files found: {len(wav_files)}")

if len(wav_files) == 0:
    raise ValueError("no .wav files found.")

# Run OpenSMILE
all_rows = []
failed_files = []

for i, wav_file in enumerate(wav_files, start=1):
    print(f"[{i}/{len(wav_files)}] Processing: {wav_file.name}")

    participant_id = wav_file.parent.name
    filename = wav_file.name

    with tempfile.NamedTemporaryFile(suffix=".csv", delete=False) as tmp:
        temp_output_csv = Path(tmp.name)

    try:
        if temp_output_csv.exists():
            temp_output_csv.unlink(missing_ok=True)

        cmd = [
            str(OPENSMILE_EXE),
            "-C", str(CONFIG_FILE),
            "-I", str(wav_file),
            "-csvoutput", str(temp_output_csv),
        ]

        result = subprocess.run(cmd, capture_output=True, text=True)

        if result.returncode != 0:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": "openSMILE return code != 0",
                "stderr": result.stderr
            })
            continue

        if not temp_output_csv.exists():
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": "no output made",
                "stderr": result.stderr
            })
            continue

        try:
            feature_df = pd.read_csv(temp_output_csv, sep=";", engine="python")
        except Exception as e:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": f"error in csv: {e}",
                "stderr": result.stderr
            })
            continue

        if feature_df.empty:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": "Output csv is leeg",
                "stderr": result.stderr
            })
            continue

        #adding information 
        row = feature_df.iloc[0].to_dict()
        row["name"] = filename
        row["participant_id"] = participant_id
        row["filename"] = filename
        row["file_path"] = str(wav_file)

        all_rows.append(row)

    except Exception as e:
        failed_files.append({
            "participant_id": participant_id,
            "filename": filename,
            "file_path": str(wav_file),
            "reason": f"error {e}",
            "stderr": ""
        })

    finally:
        if temp_output_csv.exists():
            temp_output_csv.unlink(missing_ok=True)


#Save
features_df = pd.DataFrame(all_rows)

# Kolommen netjes ordenen
preferred_front_cols = ["participant_id", "filename", "file_path", "name", "frameTime"]
remaining_cols = [col for col in features_df.columns if col not in preferred_front_cols]
features_df = features_df[[col for col in preferred_front_cols if col in features_df.columns] + remaining_cols]

features_df.to_csv(OUTPUT_FEATURES_CSV, index=False)

print("\nKlaar.")
print("number of errors in features:", len(features_df))
print("numer of errors in the files:", len(failed_files))
print("Features saves in :")
print(OUTPUT_FEATURES_CSV)

if failed_files:
    failed_df = pd.DataFrame(failed_files)
    failed_df.to_csv(OUTPUT_FAILED_CSV, index=False)
    print("\n errors saved:")
    print(OUTPUT_FAILED_CSV)


# Results
print("\nShape features dataframe:", features_df.shape)
display(features_df.head())

OpenSMILE, intensity -6dB

In [ ]:
# paths
BASE_DIR = Path(r"C:\Users\marti\Documents\Technical Medicine Master\Stages TM2\TM2-3\Technische opdracht")

OPENSMILE_EXE = BASE_DIR / "tools" / "openSMILE" / "bin" / "SMILExtract.exe"
CONFIG_FILE = BASE_DIR / "tools" / "openSMILE" / "config" / "egemaps" / "v02" / "eGeMAPSv02.conf"

AUDIO_ROOT = BASE_DIR / "TAME" / "wiener_intensity_perturbations" / "-6dB"

OUTPUT_FEATURES_CSV = BASE_DIR / "opensmile_intensity_-6dB_features.csv"
OUTPUT_FAILED_CSV = BASE_DIR / "opensmile_intensity_-6dB_failed_files.csv"

# check
print("SMILExtract exists:", OPENSMILE_EXE.exists())
print("Config exists:", CONFIG_FILE.exists())
print("Audio root exists:", AUDIO_ROOT.exists())

if not OPENSMILE_EXE.exists():
    raise FileNotFoundError(f"SMILExtract.exe not found:\n{OPENSMILE_EXE}")

if not CONFIG_FILE.exists():
    raise FileNotFoundError(f"Config file not found:\n{CONFIG_FILE}")

if not AUDIO_ROOT.exists():
    raise FileNotFoundError(f"Audio root not found:\n{AUDIO_ROOT}")

#WAV files
wav_files = sorted(AUDIO_ROOT.rglob("*.wav"))
print(f"Number of wav. files found: {len(wav_files)}")

if len(wav_files) == 0:
    raise ValueError("no .wav files found.")

# Run OpenSMILE
all_rows = []
failed_files = []

for i, wav_file in enumerate(wav_files, start=1):
    print(f"[{i}/{len(wav_files)}] Processing: {wav_file.name}")

    participant_id = wav_file.parent.name
    filename = wav_file.name

    with tempfile.NamedTemporaryFile(suffix=".csv", delete=False) as tmp:
        temp_output_csv = Path(tmp.name)

    try:
        if temp_output_csv.exists():
            temp_output_csv.unlink(missing_ok=True)

        cmd = [
            str(OPENSMILE_EXE),
            "-C", str(CONFIG_FILE),
            "-I", str(wav_file),
            "-csvoutput", str(temp_output_csv),
        ]

        result = subprocess.run(cmd, capture_output=True, text=True)

        if result.returncode != 0:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": "openSMILE return code != 0",
                "stderr": result.stderr
            })
            continue

        if not temp_output_csv.exists():
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": "no output made",
                "stderr": result.stderr
            })
            continue

        try:
            feature_df = pd.read_csv(temp_output_csv, sep=";", engine="python")
        except Exception as e:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": f"error in csv: {e}",
                "stderr": result.stderr
            })
            continue

        if feature_df.empty:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": "Output csv is leeg",
                "stderr": result.stderr
            })
            continue

        #adding information 
        row = feature_df.iloc[0].to_dict()
        row["name"] = filename
        row["participant_id"] = participant_id
        row["filename"] = filename
        row["file_path"] = str(wav_file)

        all_rows.append(row)

    except Exception as e:
        failed_files.append({
            "participant_id": participant_id,
            "filename": filename,
            "file_path": str(wav_file),
            "reason": f"error {e}",
            "stderr": ""
        })

    finally:
        if temp_output_csv.exists():
            temp_output_csv.unlink(missing_ok=True)


#Save
features_df = pd.DataFrame(all_rows)

# Kolommen netjes ordenen
preferred_front_cols = ["participant_id", "filename", "file_path", "name", "frameTime"]
remaining_cols = [col for col in features_df.columns if col not in preferred_front_cols]
features_df = features_df[[col for col in preferred_front_cols if col in features_df.columns] + remaining_cols]

features_df.to_csv(OUTPUT_FEATURES_CSV, index=False)

print("\nKlaar.")
print("number of errors in features:", len(features_df))
print("numer of errors in the files:", len(failed_files))
print("Features saves in :")
print(OUTPUT_FEATURES_CSV)

if failed_files:
    failed_df = pd.DataFrame(failed_files)
    failed_df.to_csv(OUTPUT_FAILED_CSV, index=False)
    print("\n errors saved:")
    print(OUTPUT_FAILED_CSV)


# Results
print("\nShape features dataframe:", features_df.shape)
display(features_df.head())